# Notebook 10: Comprehensive Analysis & Diagnostics

**Capstone: Bringing Everything Together**

## Overview

This notebook synthesizes results from all previous analyses:
- Traditional hypothesis tests (confidence intervals, t-tests)
- CUPED variance reduction
- Uplift modeling & heterogeneous treatment effects  
- Multi-armed bandits
- Power analysis & diagnostics

We'll create a master comparison, assess agreement across methods, evaluate assumptions, and develop a framework for recommending when to use each approach.

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Create output directory
os.makedirs('../data/outputs/nb10', exist_ok=True)


In [ ]:
# Load original data
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

# Load results from previous notebooks
print("Loading results from all analyses...\n")

# CUPED results (from notebook 7)
try:
    cuped_results = pd.read_csv('../data/outputs/nb07/nb07_cuped_results.csv')
    print("CUPED results loaded")
except:
    print("CUPED results not found - will be generated")
    cuped_results = None

# Uplift results (from notebook 8)
try:
    uplift_results = pd.read_csv('../data/outputs/nb08/nb08_uplift_results.csv')
    print("Uplift results loaded")
except:
    print("Uplift results not found")
    uplift_results = None

# Bandit results (from notebook 9)
try:
    bandit_results = pd.read_csv('../data/outputs/nb09/nb09_bandit_results.csv')
    print("Bandit results loaded")
except:
    print("Bandit results not found")
    bandit_results = None

print(f"\nDataset: {len(df)} customers across 3 segments")
print(f"\nSegment breakdown:")
print(df['segment'].value_counts())

## Method Comparison Dashboard

We'll create a master table comparing conclusions across all methods for the main outcome (conversion).

Each method provides:
- Test statistic (t-stat, p-value, Qini coefficient, etc.)
- P-value or posterior probability
- Effect size
- Confidence interval or credible interval
- Conclusion (Significant? Yes/No)

In [ ]:
# Create master comparison table for conversion outcome
# Focus on Mens Email vs No Email treatment

# Prepare data
men_email = df[df['segment'] == 'Mens E-Mail']
control = df[df['segment'] == 'No E-Mail']

y_treat = men_email['conversion'].astype(float)
y_control = control['conversion'].astype(float)

# 1. Traditional t-test (two-tailed)
t_stat, p_value_t = stats.ttest_ind(y_treat, y_control)
effect_size_t = (y_treat.mean() - y_control.mean())
se_t = np.sqrt(y_treat.var()/len(y_treat) + y_control.var()/len(y_control))
ci_t = (effect_size_t - 1.96*se_t, effect_size_t + 1.96*se_t)

# 2. Welch's t-test (doesn't assume equal variances)
t_stat_w, p_value_w = stats.ttest_ind(y_treat, y_control, equal_var=False)

# 3. Mann-Whitney U (non-parametric alternative)
u_stat, p_value_u = stats.mannwhitneyu(y_treat, y_control, alternative='two-sided')

# 4. Chi-square for proportions (binary outcome)
table = np.array([
    [y_treat.sum(), (1-y_treat).sum()],
    [y_control.sum(), (1-y_control).sum()]
])
chi2, p_value_chi2, dof, expected = stats.chi2_contingency(table)

# 5. Fisher's exact test (more conservative for small counts)
from scipy.stats import fisher_exact
odds_ratio, p_value_fisher = fisher_exact(table)

# 6. Bayesian analysis with conjugate Beta-Binomial
alpha_prior = 1
beta_prior = 1

# Posterior for treatment
alpha_post_t = alpha_prior + y_treat.sum()
beta_post_t = beta_prior + (1 - y_treat).sum()

# Posterior for control  
alpha_post_c = alpha_prior + y_control.sum()
beta_post_c = beta_prior + (1 - y_control).sum()

# Bayesian credible interval
from scipy.stats import beta as beta_dist
ci_bayes_lower_t = beta_dist.ppf(0.025, alpha_post_t, beta_post_t)
ci_bayes_upper_t = beta_dist.ppf(0.975, alpha_post_t, beta_post_t)

# Probability that treatment > control
samples_t = np.random.beta(alpha_post_t, beta_post_t, 10000)
samples_c = np.random.beta(alpha_post_c, beta_post_c, 10000)
prob_treat_better = (samples_t > samples_c).mean()

# Create comparison dataframe
comparison_methods = pd.DataFrame({
    'Method': [
        'Traditional t-test',
        "Welch's t-test", 
        'Mann-Whitney U',
        'Chi-square',
        "Fisher's Exact",
        'Bayesian Beta-Binomial'
    ],
    'Test Stat': [
        f'{t_stat:.4f}',
        f'{t_stat_w:.4f}',
        f'{u_stat:.0f}',
        f'{chi2:.4f}',
        f'{odds_ratio:.4f}',
        f'{prob_treat_better:.4f}'
    ],
    'P-Value/Prob': [
        f'{p_value_t:.4f}',
        f'{p_value_w:.4f}',
        f'{p_value_u:.4f}',
        f'{p_value_chi2:.4f}',
        f'{p_value_fisher:.4f}',
        f'P(T>C)={prob_treat_better:.4f}'
    ],
    'Effect Size': [
        f'{effect_size_t:.4f}',
        f'{effect_size_t:.4f}',
        f'r={u_stat/(len(y_treat)*len(y_control)):.4f}',
        f'OR={odds_ratio:.4f}',
        f'OR={odds_ratio:.4f}',
        f'DP={effect_size_t:.4f}'
    ],
    'CI Lower': [
        f'{ci_t[0]:.4f}',
        f'{ci_t[0]:.4f}',
        '---',
        '---',
        '---',
        f'{effect_size_t - 1.96*se_t:.4f}'
    ],
    'CI Upper': [
        f'{ci_t[1]:.4f}',
        f'{ci_t[1]:.4f}',
        '---',
        '---',
        '---',
        f'{effect_size_t + 1.96*se_t:.4f}'
    ],
    'Significant': [
        'Yes' if p_value_t < 0.05 else 'No',
        'Yes' if p_value_w < 0.05 else 'No',
        'Yes' if p_value_u < 0.05 else 'No',
        'Yes' if p_value_chi2 < 0.05 else 'No',
        'Yes' if p_value_fisher < 0.05 else 'No',
        'Yes' if abs(prob_treat_better - 0.5) > 0.05 else 'No'
    ]
})

print("\n=== COMPREHENSIVE METHOD COMPARISON (CONVERSION) ===\n")
print(comparison_methods.to_string(index=False))

# Save
comparison_methods.to_csv('../data/outputs/nb10/nb10_method_comparison_conversion.csv', index=False)

In [ ]:
# Repeat for spend (continuous outcome)
y_spend_treat = men_email['spend']
y_spend_control = control['spend']

# Parametric tests
t_stat_spend, p_value_spend_t = stats.ttest_ind(y_spend_treat, y_spend_control, equal_var=False)
effect_size_spend = y_spend_treat.mean() - y_spend_control.mean()
se_spend = np.sqrt(y_spend_treat.var()/len(y_spend_treat) + y_spend_control.var()/len(y_spend_control))
ci_spend = (effect_size_spend - 1.96*se_spend, effect_size_spend + 1.96*se_spend)

# Non-parametric
u_stat_spend, p_value_spend_u = stats.mannwhitneyu(y_spend_treat, y_spend_control, alternative='two-sided')

# Bayesian (normal likelihood)
mean_t = y_spend_treat.mean()
std_t = y_spend_treat.std()
n_t = len(y_spend_treat)

mean_c = y_spend_control.mean()
std_c = y_spend_control.std()
n_c = len(y_spend_control)

# Posterior for difference
df_bayes = n_t + n_c - 2
t_crit = stats.t.ppf(0.975, df_bayes)

comparison_spend = pd.DataFrame({
    'Method': [
        "Welch's t-test",
        'Mann-Whitney U',
        'Bayesian Normal'
    ],
    'Test Stat': [
        f'{t_stat_spend:.4f}',
        f'{u_stat_spend:.0f}',
        f'{effect_size_spend:.2f}'
    ],
    'P-Value': [
        f'{p_value_spend_t:.4f}',
        f'{p_value_spend_u:.4f}',
        f'{p_value_spend_t:.4f}'
    ],
    'Effect Size': [
        f'${effect_size_spend:.2f}',
        f'r={u_stat_spend/(len(y_spend_treat)*len(y_spend_control)):.4f}',
        f'${effect_size_spend:.2f}'
    ],
    'CI Lower': [
        f'${ci_spend[0]:.2f}',
        '---',
        f'${ci_spend[0]:.2f}'
    ],
    'CI Upper': [
        f'${ci_spend[1]:.2f}',
        '---',
        f'${ci_spend[1]:.2f}'
    ],
    'Significant': [
        'Yes' if p_value_spend_t < 0.05 else 'No',
        'Yes' if p_value_spend_u < 0.05 else 'No',
        'Yes' if p_value_spend_t < 0.05 else 'No'
    ]
})

print("\n=== COMPREHENSIVE METHOD COMPARISON (SPEND) ===\n")
print(comparison_spend.to_string(index=False))

comparison_spend.to_csv('../data/outputs/nb10/nb10_method_comparison_spend.csv', index=False)

## Agreement Analysis

Do different statistical methods agree? Where do they disagree?

**High agreement** -> Results are robust
**Low agreement** -> Results depend on method choice (be cautious)

In [ ]:
# Concordance matrix: pairwise agreement between methods
methods_list = comparison_methods['Method'].tolist()
p_values_list = [p_value_t, p_value_w, p_value_u, p_value_chi2, p_value_fisher]
conclusions = [p < 0.05 for p in p_values_list]

# Create concordance matrix
n_methods = len(p_values_list)
concordance_matrix = np.zeros((n_methods, n_methods))

for i in range(n_methods):
    for j in range(n_methods):
        if i == j:
            concordance_matrix[i, j] = 1.0
        else:
            # Both significant or both not significant
            if conclusions[i] == conclusions[j]:
                concordance_matrix[i, j] = 1.0
            else:
                concordance_matrix[i, j] = 0.0

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))
method_short = ['t-test', "Welch's", 'Mann-Whitney', 'Chi-square', "Fisher's"]
sns.heatmap(concordance_matrix, annot=True, fmt='.2f', cmap='RdYlGn', 
            xticklabels=method_short, yticklabels=method_short,
            cbar_kws={'label': 'Agreement'}, ax=ax, vmin=0, vmax=1)
ax.set_title('Method Concordance Matrix (Conversion Outcome)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/outputs/nb10/nb10_method_agreement.png', dpi=150, bbox_inches='tight')
plt.show()

agreement_pct = np.mean(concordance_matrix[np.triu_indices_from(concordance_matrix, k=1)])
print(f"\nOverall agreement across methods: {agreement_pct:.1%}")
print(f"Interpretation: High agreement means robust results.")

## Statistical Power Analysis

**Power:** The probability of detecting a true effect if it exists.

Post-hoc power analysis tells us: "Given our sample size and observed effect, what was our ability to detect effects of various sizes?"

**Sample size calculator:** How many customers would we need for 80% power for different effect sizes?

In [ ]:
from scipy.stats import norm

# Post-hoc power analysis for conversion
p_treat = y_treat.mean()
p_control = y_control.mean()
n_treat = len(y_treat)
n_control = len(y_control)

# Effect size for proportions (Cohens h)
cohens_h = 2 * (np.arcsin(np.sqrt(p_treat)) - np.arcsin(np.sqrt(p_control)))
print(f"=== POST-HOC POWER ANALYSIS (Conversion) ===")
print(f"\nObserved effect size (Cohens h): {cohens_h:.4f}")
print(f"Treatment conversion rate: {p_treat:.4f}")
print(f"Control conversion rate: {p_control:.4f}")
print(f"Sample size per group: {n_treat}")

# Post-hoc power
z_alpha = norm.ppf(0.975)
ncp = np.sqrt(n_treat + n_control) / 2 * cohens_h
post_hoc_power = 1 - norm.cdf(z_alpha - ncp)

print(f"\nPost-hoc Power (alpha=0.05, two-tailed): {post_hoc_power:.1%}")

# Sample size needed for different effect sizes
print(f"\n=== SAMPLE SIZE CALCULATOR ===")
print(f"\nTo achieve 80% power to detect effect sizes:")

effect_sizes_h = np.array([0.1, 0.2, 0.3, 0.5])
power_target = 0.80
z_beta = norm.ppf(power_target)

sample_size_table = []
for h in effect_sizes_h:
    n_needed = (z_alpha + z_beta)**2 / (2 * h**2)
    sample_size_table.append({
        'Effect Size (h)': f'{h:.2f}',
        'Sample/Group': int(n_needed),
        'Total': int(n_needed * 2)
    })

sample_size_df = pd.DataFrame(sample_size_table)
print(sample_size_df.to_string(index=False))

# Visualize power curve
alphas = np.linspace(0.01, 0.20, 100)
powers_by_alpha = []
for a in alphas:
    ncp_a = np.sqrt(n_treat + n_control) / 2 * a
    power_a = 1 - norm.cdf(z_alpha - ncp_a)
    powers_by_alpha.append(power_a)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(alphas, powers_by_alpha, linewidth=2.5, color='#1f77b4')
ax1.axhline(0.80, color='red', linestyle='--', linewidth=2, label='80% Power')
ax1.axhline(0.90, color='orange', linestyle='--', linewidth=2, label='90% Power')
ax1.axvline(cohens_h, color='green', linestyle=':', linewidth=2.5, label=f'Observed (h={cohens_h:.4f})')
ax1.fill_between(alphas, 0, powers_by_alpha, alpha=0.2, color='#1f77b4')
ax1.set_xlabel("Effect Size (Cohens h)", fontsize=11)
ax1.set_ylabel('Statistical Power', fontsize=11)
ax1.set_title('Power Curve for Conversion Analysis', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 1)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

effect_sizes = np.linspace(0.05, 0.5, 50)
sample_sizes = (z_alpha + z_beta)**2 / (2 * effect_sizes**2)

ax2.plot(effect_sizes, sample_sizes, linewidth=2.5, color='#ff7f0e')
ax2.axhline(n_treat, color='green', linestyle=':', linewidth=2, label=f'Current n={n_treat}')
ax2.set_xlabel("Effect Size (Cohens h)", fontsize=11)
ax2.set_ylabel('Sample Size per Group', fontsize=11)
ax2.set_title('Sample Size Needed (80% Power)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb10/nb10_power_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Assumption Diagnostics

Statistical tests rely on assumptions:

1. **Normality** (for t-tests): Outcomes normally distributed
2. **Homogeneity of Variance**: Variance is same in both groups
3. **Independence**: Observations are independent
4. **Sample Ratio Mismatch (SRM)**: Did randomization work?

In [ ]:
# Normality diagnostics for spend outcome
from scipy.stats import shapiro

print("=== NORMALITY TESTS (Spend Outcome) ===\n")

# Shapiro-Wilk test
stat_t, p_shapiro_t = shapiro(y_spend_treat)
stat_c, p_shapiro_c = shapiro(y_spend_control)

print(f"Shapiro-Wilk Test:")
print(f"  Treatment: W={stat_t:.4f}, p={p_shapiro_t:.4f}")
print(f"  Control: W={stat_c:.4f}, p={p_shapiro_c:.4f}")

# Q-Q plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

stats.probplot(y_spend_treat, dist="norm", plot=axes[0])
axes[0].set_title("Q-Q Plot: Treatment Group (Spend)", fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

stats.probplot(y_spend_control, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot: Control Group (Spend)", fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb10/nb10_normality_qq_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Sensitivity Analysis

How robust are our conclusions to different assumptions and choices?

1. **Significance level**: What if we used alpha=0.10 instead of 0.05?
2. **One-sided vs two-sided**: What if we only cared about increase?
3. **Outlier removal**: What if we excluded extreme values?

In [ ]:
# Sensitivity analysis for conversion
alpha_values = [0.01, 0.05, 0.10]
sensitivity_results = []

for alpha in alpha_values:
    z_crit = norm.ppf(1 - alpha/2)
    ci_width = z_crit * se_t
    p_sig = p_value_t < alpha
    
    sensitivity_results.append({
        'Significance Level': f'{alpha:.2f}',
        'Critical t': f'{z_crit:.3f}',
        'Significant': 'Yes' if p_sig else 'No',
        'Effect': f'{effect_size_t:.4f}'
    })

# One-sided
p_one_sided = p_value_t / 2
sensitivity_results.append({
    'Significance Level': '0.05 (1-sided)',
    'Critical t': '1.645',
    'Significant': 'Yes' if p_one_sided < 0.05 else 'No',
    'Effect': f'{effect_size_t:.4f}'
})

# Outlier removal
y_treat_trim = y_treat[(y_treat >= y_treat.quantile(0.01)) & (y_treat <= y_treat.quantile(0.99))]
y_control_trim = y_control[(y_control >= y_control.quantile(0.01)) & (y_control <= y_control.quantile(0.99))]
t_trim, p_trim = stats.ttest_ind(y_treat_trim, y_control_trim, equal_var=False)
effect_trim = y_treat_trim.mean() - y_control_trim.mean()

sensitivity_results.append({
    'Significance Level': '0.05 (trimmed 1%)',
    'Critical t': f'{t_trim:.3f}',
    'Significant': 'Yes' if p_trim < 0.05 else 'No',
    'Effect': f'{effect_trim:.4f}'
})

sensitivity_df = pd.DataFrame(sensitivity_results)
print("\n=== SENSITIVITY ANALYSIS ===\n")
print(sensitivity_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))

scenarios = ['alpha=0.01', 'alpha=0.05', 'alpha=0.10', 'One-sided', 'Trimmed 1%']
effects = [effect_size_t, effect_size_t, effect_size_t, effect_size_t, effect_trim]
p_values = [p_value_t, p_value_t, p_value_t, p_one_sided, p_trim]
significant = ['Yes' if p < 0.05 else 'No' for p in p_values]

colors_sensitivity = ['#2ca02c' if sig == 'Yes' else '#d62728' for sig in significant]
ax.barh(scenarios, effects, color=colors_sensitivity, alpha=0.7, edgecolor='black')

ax.set_xlabel('Effect Size (Conversion Rate Difference)', fontsize=11)
ax.set_title('Sensitivity Analysis: Effect Robustness', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', linestyle='-', linewidth=1)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb10/nb10_sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Practical vs Statistical Significance

**Statistical significance** is not the same as **business significance**.

A difference can be:
- Statistically significant AND practically meaningful
- Statistically significant BUT negligible for business
- NOT statistically significant BUT practically important

In [ ]:
# Business impact analysis for conversion
print("=== BUSINESS IMPACT ANALYSIS ===\n")

email_cost = 0.50
conversion_value = 50.0
n_customers = 1_000_000
percent_receiving_email = 0.30

customers_receiving = int(n_customers * percent_receiving_email)
customers_control = n_customers - customers_receiving

conversions_email = customers_receiving * p_treat
conversions_control = customers_receiving * p_control

incremental_conversions = conversions_email - conversions_control
total_spend = customers_receiving * email_cost
total_incremental_revenue = incremental_conversions * conversion_value
net_profit = total_incremental_revenue - total_spend

print(f"Scenario: Send emails to {percent_receiving_email:.0%} of {n_customers/1e6:.1f}M customers")
print(f"\nObserved Effect:")
print(f"  Treatment conversion rate: {p_treat:.3f} ({p_treat*100:.1f}%)")
print(f"  Control conversion rate: {p_control:.3f} ({p_control*100:.1f}%)")
print(f"  Incremental lift: {effect_size_t*100:.2f} percentage points")

print(f"\nFinancial Impact:")
print(f"  Customers emailed: {customers_receiving:,.0f}")
print(f"  Total email cost: ${total_spend:,.0f}")
print(f"  Incremental conversions: {incremental_conversions:,.0f}")
print(f"  Incremental revenue: ${total_incremental_revenue:,.0f}")
print(f"  Net profit: ${net_profit:,.0f}")

print(f"\nROI Analysis:")
roi = (net_profit / total_spend) * 100 if total_spend > 0 else 0
print(f"  ROI: {roi:.0f}%")

print(f"\nDecision:")
if net_profit > 0 and p_value_t < 0.05:
    print(f"  PROFITABLE AND STATISTICALLY SIGNIFICANT - Implement")
elif net_profit > 0:
    print(f"  PROFITABLE BUT NOT SIGNIFICANT - Run larger test")
else:
    print(f"  NOT PROFITABLE - Do not implement")

In [ ]:
# Save master results
results_summary = pd.DataFrame({
    'Test': ['Conversion (t-test)', 'Spend (Welch)', 'Conversion (Chi-sq)'],
    'Effect Size': [f'{effect_size_t:.4f}', f'{effect_size_spend:.2f}', f'{odds_ratio:.4f}'],
    'P-Value': [f'{p_value_t:.4f}', f'{p_value_spend_t:.4f}', f'{p_value_chi2:.4f}'],
    'Significant': [
        'Yes' if p_value_t < 0.05 else 'No',
        'Yes' if p_value_spend_t < 0.05 else 'No',
        'Yes' if p_value_chi2 < 0.05 else 'No'
    ]
})

results_summary.to_csv('../data/outputs/nb10/nb10_comprehensive_results_summary.csv', index=False)

print("\nFiles saved:")
print("1. method_comparison_conversion.csv")
print("2. method_comparison_spend.csv")
print("3. comprehensive_results_summary.csv")
print("\nVisualizations:")
print("4. method_agreement.png")
print("5. power_analysis.png")
print("6. normality_qq_plots.png")
print("7. sensitivity_analysis.png")

## Key Takeaways

### Notebook Summary

This notebook synthesized all A/B testing methods:
1. **Traditional t-tests** - statistically rigorous
2. **CUPED** - reduces variance with covariates
3. **Uplift modeling** - identifies individual treatment effects
4. **Multi-armed bandits** - adaptive allocation
5. **Bayesian approach** - sequential analysis

### Method Agreement
High concordance across multiple statistical tests indicates robust results.

### Power Analysis
Post-hoc power reveals our ability to detect effects of various sizes.

### Assumptions
Diagnostic tests ensure our statistical conclusions are valid.

### Business Impact
ROI analysis connects statistical significance to business value.

### Recommendations

Choose your method based on:
- **Outcome type** (binary vs continuous)
- **Decision timeline** (one-time vs ongoing)
- **Cost constraints** (sample size vs quality)
- **Personalization needs** (average vs individual effects)